<a href="https://colab.research.google.com/github/HTKhuongNinh-FPTU/DS-AI-Project/blob/main/DataProcess%26EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tập dữ liệu của dự án này chỉ lấy dữ liệu của thành phố có ID 03 từ tập dữ liệu train 4.5M rows. Vì dữ liệu tổng quá lớn nên chỉ thực hiện ở 1 quy mô dữ liệu nhỏ hơn.

**PHASE 1 (DATA INGESTION & OVERVIEW)**

In [1]:
from datasets import load_dataset
import pandas as pd

# 1. Tải dataset
print("Đang tải dữ liệu về máy...")
dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K", split='train')
# 2. Lọc dữ liệu bằng bộ lọc của dataset
print("Đang lọc City ID 3...")
df_city_3 = dataset.filter(lambda example: example['city_id'] == 3)
print(f"Đã hoàn thành! Có {len(df_city_3)} dòng dữ liệu cho City ID 3.")

Đang tải dữ liệu về máy...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.87k [00:00<?, ?B/s]

data/train.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

data/eval.parquet:   0%|          | 0.00/8.44M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/350000 [00:00<?, ? examples/s]

Đang lọc City ID 3...


Filter:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Đã hoàn thành! Có 238320 dòng dữ liệu cho City ID 3.


Overview

In [3]:
# Convert to Pandas DataFrame for data processing
print("\nConverting to Pandas DataFrame...")
df_city_3 = pd.DataFrame(df_city_3)

# Check size and preview data
print(f"Dataset size: {df_city_3.shape}")
df_city_3.head()


Converting to Pandas DataFrame...
Dataset size: (238320, 19)


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,hours_sale,stock_hour6_22_cnt,hours_stock_status,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level
0,3,70,0,5,6,65,157,2024-03-28,1.80,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7, 1.1, 0.0, ...",15,"[1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, ...",0.914,0,0,0.0142,14.32,31.25,1.57
1,3,70,0,5,6,65,157,2024-03-29,1.50,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3, ...",9,"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, ...",0.897,0,0,0.4079,14.60,29.95,1.57
2,3,70,0,5,6,65,157,2024-03-30,1.80,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6, 0.0, 0.6, ...",11,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, ...",0.897,1,0,0.6128,14.68,31.86,1.34
3,3,70,0,5,6,65,157,2024-03-31,1.78,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6, 0.0, ...",3,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.903,1,0,0.3063,16.94,35.42,1.38
4,3,70,0,5,6,65,157,2024-04-01,1.80,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3, 0.0, ...",9,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, ...",0.897,0,0,0.7007,16.21,39.26,1.23


**PHASE 2 (DATA SELECTION & CLEANING)**

Loại bỏ city_id column

In [4]:
columns_to_drop = ['city_id']
df_city_3 = df_city_3.drop(columns=columns_to_drop, errors='ignore')
print(f"Dropped columns: {columns_to_drop}")
print(f"New DataFrame shape: {df_city_3.shape}")

Dropped columns: ['city_id']
New DataFrame shape: (238320, 18)


Data Validation


In [6]:
print("Checking for Missing Values:")
missing_values = df_city_3.isnull().sum()
missing_percentage = (df_city_3.isnull().sum() / len(df_city_3)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing Percentage': missing_percentage})
print(missing_df[missing_df['Missing Count'] > 0])
if missing_df[missing_df['Missing Count'] > 0].empty:
    print("No missing values found.\n")
else:
    print("\nColumns with missing values found. Consider imputation or removal.\n")

print("Checking for Duplicate Rows:")
list_cols = [col for col in df_city_3.columns if isinstance(df_city_3[col].iloc[0], list) or isinstance(df_city_3[col].iloc[0], tuple)]

subset_cols_for_duplicate_check = [col for col in df_city_3.columns if col not in list_cols]
duplicate_rows = df_city_3.duplicated(subset=subset_cols_for_duplicate_check).sum()
if duplicate_rows == 0:
    print("No duplicate rows found (excluding list-type columns).\n")
else:
    print(f"Found {duplicate_rows} duplicate rows (excluding list-type columns). Consider removing them.\n")

print("Checking Data Types and Info:")
df_city_3.info()

Checking for Missing Values:
Empty DataFrame
Columns: [Missing Count, Missing Percentage]
Index: []
No missing values found.

Checking for Duplicate Rows:
No duplicate rows found (excluding list-type columns).

Checking Data Types and Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238320 entries, 0 to 238319
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   store_id             238320 non-null  int64  
 1   management_group_id  238320 non-null  int64  
 2   first_category_id    238320 non-null  int64  
 3   second_category_id   238320 non-null  int64  
 4   third_category_id    238320 non-null  int64  
 5   product_id           238320 non-null  int64  
 6   dt                   238320 non-null  object 
 7   sale_amount          238320 non-null  float64
 8   hours_sale           238320 non-null  object 
 9   stock_hour6_22_cnt   238320 non-null  int64  
 10  hours_stock_status   238320 non

**PHASE 3 (EDA)**

Create file

In [ ]:
output_filename = "freshretailnet_city_03_dataset.csv"
df_city_3.to_csv(output_filename, index=False)
print(f"DataFrame saved to {output_filename}")

DataFrame saved to freshretailnet_city_03_dataset.csv


END